# Session 3 — Abundance across PTEN positions

Summarize the missense measurements at each of the 403 positions: how much data there is, and what the scores look like. This builds on your Session 2 parsing.

**Aims** (from the end of the Session 3 slides), using all *checked* missense variants (those whose WT residue matched the reference):
1. Describe coverage (number of observed missense variants) across all 403 positions, including positions with no measured substitutions.
2. Compute score statistics for each position with at least `minimum_variants = 5` included substitutions. Keep missing scores distinct from zero.
3. Write test functions to verify that the code computing abundance and score statistics works as expected and flags or rejects unexpected information.
4. Visualize coverage and score statistics across the PTEN protein sequence.

**For each analysis, be ready to explain:** what you did, how the code works, and how you checked it.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

### Functions
All the functions used in this notebook are defined here, once. `get_checked_missense` is the Session 2 parsing and checking turned into a function. The contracts and tests for these functions are in Aim 3.

In [ ]:
# 3-letter amino acid code -> 1-letter code
letter = {"Ala": "A", "Arg": "R", "Asn": "N", "Asp": "D", "Cys": "C", "Gln": "Q", "Glu": "E", "Gly": "G",
          "His": "H", "Ile": "I", "Leu": "L", "Lys": "K", "Met": "M", "Phe": "F", "Pro": "P", "Ser": "S",
          "Thr": "T", "Trp": "W", "Tyr": "Y", "Val": "V"}


def read_fasta(path):
    """Return the protein sequence in a FASTA file as one string."""
    lines = path.read_text().splitlines()
    return "".join(line.strip() for line in lines if not line.startswith(">"))


def get_checked_missense(table, ref_seq):
    """Return the missense rows of table, with new columns wt, pos, and mut.

    Raises ValueError if a label cannot be parsed, is not a real substitution,
    sits outside the protein, has a WT residue that does not match ref_seq, or
    is duplicated. Missing scores are kept as NaN.
    """
    df = table[table["record_type"] == "missense"].copy()

    # "p.His75Asp" -> wt "His", site 75, mutant "Asp"
    parts = df["hgvs_pro"].str.extract(r"^p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})$")
    if parts[0].isna().any():
        raise ValueError("unparseable label(s): " + str(df.loc[parts[0].isna(), "hgvs_pro"].tolist()[:5]))
    df["wt"], df["pos"], df["mut"] = parts[0], parts[1].astype(int), parts[2]

    if not df["mut"].isin(list(letter)).all() or (df["mut"] == df["wt"]).any():
        raise ValueError("not a missense substitution")
    if not df["pos"].between(1, len(ref_seq)).all():
        raise ValueError("site outside the protein")

    # sites count from 1, so label the reference letters 1, 2, 3, ...
    ref_by_site = pd.Series(list(ref_seq), index=range(1, len(ref_seq) + 1))
    if (df["wt"].map(letter) != df["pos"].map(ref_by_site)).any():
        raise ValueError("WT residue does not match the reference")
    if df["hgvs_pro"].duplicated().any():
        raise ValueError("duplicate labels")

    df["score"] = pd.to_numeric(df["score"])
    return df[["hgvs_pro", "wt", "pos", "mut", "score", "expts"]]


def coverage(missense, n_sites):
    """Number of missense variants at each site 1..n_sites (0 where none were measured)."""
    counts = missense.groupby("pos").size()
    return counts.reindex(range(1, n_sites + 1), fill_value=0)


def site_stats(missense, n_sites, minimum_variants=5):
    """Score statistics for each site 1..n_sites.

    n_variants counts the variants with a score (0 where there are none).
    The statistics are NaN, never 0, where n_variants < minimum_variants.
    """
    scored = missense.dropna(subset=["score"])
    stats = scored.groupby("pos")["score"].agg(["count", "mean", "median", "std", "min", "max"])
    stats = stats.reindex(range(1, n_sites + 1))
    stats["count"] = stats["count"].fillna(0).astype(int)
    stats.loc[stats["count"] < minimum_variants, ["mean", "median", "std", "min", "max"]] = np.nan
    return stats.rename(columns={"count": "n_variants"})

Locate the data. This notebook assumes Jupyter is running with its working directory set to this notebook's folder (`code/my_code`), so the project root is two levels up. If a path check prints `False`, check `Path.cwd()`.

In [ ]:
project_root = Path.cwd().parent.parent
data_dir = project_root / "data" / "raw_data"

abundance_path = data_dir / "derived" / "pten-variant-abundance.csv"
fasta_path = data_dir / "derived" / "pten-reference-protein.fasta"

abundance_path.exists(), fasta_path.exists()

In [ ]:
minimum_variants = 5   # a site needs at least this many scored variants to get statistics

abundance = pd.read_csv(abundance_path)
ref_seq = read_fasta(fasta_path)
n_sites = len(ref_seq)   # take the length from the sequence, not from a typed number
n_sites

## Preparing the missense table

Reuse your Session 2 work: which rows are missense, what site is each, and did its WT residue match the reference? Decide what counts as "included" and write that rule down.

*Inclusion rule used below:* a row is included if its `record_type` is `missense`, its label parses into WT residue, site, and mutant, the site is inside the protein, the WT residue matches the reference, and the label is not duplicated. A variant with a missing score stays in the table (it counts toward coverage) but is not counted when computing site statistics.

In [ ]:
missense = get_checked_missense(abundance, ref_seq)
missense.head()

**Check:** how many missense variants are included? Compare with the counts in `manifest.json`. Does any site fall outside 1 to the reference length? Are there duplicate labels?

In [ ]:
len(missense), missense["pos"].between(1, n_sites).all(), missense["hgvs_pro"].duplicated().any()

## Aim 1. Coverage across all 403 positions

Questions: What is coverage here, precisely? What should happen at a position with *no* observed substitutions: is it a row with 0, or a missing row? Why does that matter for a plot later?

In [ ]:
cov = coverage(missense, n_sites)
cov.head()

**Summaries to report:** how many positions have no variants, 1 to 4 variants, and 5 or more? Which positions have none?

In [ ]:
print("sites with no variants:", (cov == 0).sum())
print("sites with 1 to", minimum_variants - 1, "variants:", ((cov >= 1) & (cov < minimum_variants)).sum())
print("sites with", minimum_variants, "or more variants:", (cov >= minimum_variants).sum())
print("sites with no variants:", cov[cov == 0].index.tolist())

**Check:** the coverage counts should add up to the number of variants, and there should be one entry per site.

In [ ]:
cov.sum() == len(missense), len(cov) == n_sites

**Presentation notes**
- What I did:
- How the code works:
- How I checked:

## Aim 2. Score statistics per position

Decisions to record:
- Which statistics will you compute, and why? (Session 4's discussion asks this.) The code below computes the number of variants, mean, median, standard deviation, minimum, and maximum. Median and mean can differ when a site has a few extreme variants.
- A position with too few variants: what value should its statistics take? How is "not enough data" different from "a score of zero", and how is either different from "no variants observed"? In the code, `n_variants` is always a real count (0 when nothing was observed) and the statistics are `NaN` below the threshold.

*Your choices and reasoning:*

In [ ]:
stats = site_stats(missense, n_sites, minimum_variants)
stats.head(10)

**Check:** how many positions pass the threshold? Pick one position by hand and compare your statistics against a calculation on its rows.

In [ ]:
print("sites with statistics:", stats["median"].notna().sum(), "of", n_sites)

site = 75
by_hand = missense.loc[missense["pos"] == site, "score"]
print("by hand, site", site, ":", len(by_hand), "variants, median", by_hand.median())
stats.loc[site]

**Presentation notes**
- What I did:
- How the code works:
- How I checked:

## Aim 3. Test functions

Following the course's habit, each function starts from a **contract**: what input it accepts, what it returns, and what it does *not* change. Then there are known-answer tests on a tiny table built by hand, plus tests chosen to expose plausible mistakes (for example, treating a missing score as zero) and tests that bad input is rejected.

*Contract for `get_checked_missense(table, ref_seq)`:*
- Input: a table with columns `hgvs_pro`, `record_type`, `score`, `expts`, and a reference sequence string.
- Output: a new table of only the missense rows, with `wt`, `pos`, and `mut` columns. The input table is not changed.
- Raises `ValueError` if a label cannot be parsed, is not a real substitution (unknown amino acid, or mutant equal to WT), has a site outside the protein, has a WT residue that does not match the reference, or is duplicated. Missing scores are allowed.

*Contract for `coverage(missense, n_sites)`:*
- Input: the checked missense table and the number of sites.
- Output: an integer count for every site from 1 to `n_sites`, with 0 where nothing was measured.

*Contract for `site_stats(missense, n_sites, minimum_variants)`:*
- Input: the checked missense table, the number of sites, and the threshold.
- Output: one row per site from 1 to `n_sites`. `n_variants` is a real count of variants with a score (0 allowed). The statistics are `NaN` where `n_variants < minimum_variants`. A missing score is never treated as 0.

The tests are defined here, next to the aim they belong to, and not in the Functions cell.

### Tests

In [ ]:
toy_ref = "AGLWM"   # five sites: Ala, Gly, Leu, Trp, Met


def toy_table():
    """Hand-made table. Site 1 has 5 variants, site 2 has 2, site 4 has 6, sites 3 and 5 have none."""
    rows = [("p.Ala1Gly", 1.0), ("p.Ala1Ser", 1.2), ("p.Ala1Thr", 0.8), ("p.Ala1Val", 1.1), ("p.Ala1Cys", 0.9),
            ("p.Gly2Ala", 0.5), ("p.Gly2Ser", 0.6),
            ("p.Trp4Ala", 2.0), ("p.Trp4Cys", 2.2), ("p.Trp4Gly", 1.8), ("p.Trp4Ser", 2.1),
            ("p.Trp4Thr", 1.9), ("p.Trp4Val", 2.0)]
    table = pd.DataFrame(rows, columns=["hgvs_pro", "score"])
    table["record_type"] = "missense"
    table["expts"] = 4.0
    # rows that must not count as missense
    other = pd.DataFrame({"hgvs_pro": ["p.Leu3=", "p.Trp4Ter"], "score": [1.0, 0.0],
                          "record_type": ["synonymous", "nonsense"], "expts": 4.0})
    return pd.concat([table, other], ignore_index=True)


def test_coverage_counts():
    cov_toy = coverage(get_checked_missense(toy_table(), toy_ref), 5)
    assert list(cov_toy) == [5, 2, 0, 6, 0]   # includes sites with no variants; ignores synonymous and nonsense


def test_stats_values():
    stats_toy = site_stats(get_checked_missense(toy_table(), toy_ref), 5, 5)
    assert np.isclose(stats_toy.loc[1, "median"], 1.0) and np.isclose(stats_toy.loc[1, "mean"], 1.0)
    assert np.isclose(stats_toy.loc[4, "median"], 2.0) and np.isclose(stats_toy.loc[4, "max"], 2.2)


def test_below_threshold_is_nan_not_zero():
    stats_toy = site_stats(get_checked_missense(toy_table(), toy_ref), 5, 5)
    assert stats_toy.loc[2, "n_variants"] == 2 and np.isnan(stats_toy.loc[2, "median"])   # too few variants
    assert stats_toy.loc[3, "n_variants"] == 0 and np.isnan(stats_toy.loc[3, "median"])   # none observed


def test_threshold_is_inclusive():
    checked = get_checked_missense(toy_table(), toy_ref)
    assert not np.isnan(site_stats(checked, 5, 2).loc[2, "median"])   # 2 variants, threshold 2
    assert np.isnan(site_stats(checked, 5, 3).loc[2, "median"])       # 2 variants, threshold 3


def test_missing_score_is_not_zero():
    table = toy_table()
    table.loc[0, "score"] = np.nan   # drop one site-1 variant (its score was 1.0)
    stats_toy = site_stats(get_checked_missense(table, toy_ref), 5, 4)
    assert stats_toy.loc[1, "n_variants"] == 4
    assert np.isclose(stats_toy.loc[1, "mean"], 1.0)   # would be 0.8 if the NaN were counted as 0


def test_bad_input_is_rejected():
    bad_labels = ["p.Gly1Ala",    # WT residue does not match the reference (site 1 is Ala)
                  "p.Ala99Gly",   # site outside the protein
                  "p.Ala1Ter",    # a stop, not a missense substitution
                  "p.Ala1Ala",    # mutant equal to WT
                  "p.Ala1Xyz",    # unknown amino acid
                  "not_a_label",  # cannot be parsed
                  "p.Ala1Ser"]    # duplicate of another row
    for label in bad_labels:
        table = toy_table()
        table.loc[0, "hgvs_pro"] = label
        try:
            get_checked_missense(table, toy_ref)
        except ValueError:
            continue
        raise AssertionError("should have been rejected: " + label)


for test in [test_coverage_counts, test_stats_values, test_below_threshold_is_nan_not_zero,
             test_threshold_is_inclusive, test_missing_score_is_not_zero, test_bad_input_is_rejected]:
    test()
    print("passed:", test.__name__)

### Audit
What inputs are *outside* your contract? What does your code do with them: reject, flag, or silently accept? Does a passing test prove more than the specific thing it checks?

*Your audit:*

**Presentation notes**
- What I did:
- How the code works:
- How I checked:

## Aim 4. Visualize coverage and score statistics across the sequence

Questions: What plot will make gaps in coverage and gaps in statistics visible instead of hiding them? What patterns do you see along the sequence? What could explain them, keeping in mind that this assay measures abundance only?

In the figure below, the top panel shows coverage with the threshold. The bottom panel shows each site's median score, with a gray bar for its minimum-to-maximum range, and a red tick for every site that has no statistic.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

ax1.bar(cov.index, cov.values, width=1.0, color="steelblue")
ax1.axhline(minimum_variants, color="red", ls="--", lw=1, label="minimum_variants = " + str(minimum_variants))
ax1.set_ylabel("missense variants per site")
ax1.set_title("PTEN coverage and abundance by position")
ax1.legend(loc="upper right")

ax2.vlines(stats.index, stats["min"], stats["max"], color="lightgray", lw=1, label="min to max")
ax2.scatter(stats.index, stats["median"], s=10, color="black", label="median score")
ax2.axhline(1, color="gray", ls="--", lw=1)   # WT reference
excluded = stats.index[stats["median"].isna()]
ax2.scatter(excluded, np.full(len(excluded), stats["min"].min() - 0.1), marker="|", color="red", s=30,
            label="no statistic (too few or no variants)")
ax2.set_xlabel("PTEN amino acid position")
ax2.set_ylabel("abundance score (WT = 1, dashed)")
ax2.set_xlim(0, n_sites + 1)
ax2.legend(loc="lower right", ncol=3)

plt.tight_layout()
plt.show()

**Figure checklist** (from the Session 4 slides)
- Are the labels descriptive and legible, with units where relevant?
- Is the WT reference (score = 1) visible or explained? What is "high" and what is "low"?
- Does `figsize` match the size the figure will be shown at?
- Do color and shape mean the same thing in every panel and figure?
- Is the palette right for the data: distinct colors for unordered groups, sequential for ordered amounts, diverging only around a meaningful center?
- Are colorbars labeled with visible scale marks? Do gridlines stay quiet?

*Patterns I observe:*

**Presentation notes**
- What I did:
- How the code works:
- How I checked:

## Reproducibility record

Run this last, after restarting the kernel and running all cells.

In [ ]:
import session_info
session_info.show(dependencies=True)